In [ ]:
# ─── 1. CÀI ĐẶT MÔI TRƯỜNG ───────────────────────────────────────────────────
!pip uninstall -y transformers peft accelerate -q
!pip install -q \
    transformers==4.41.2 \
    peft==0.11.1 \
    accelerate==0.30.1 \
    datasets \
    scikit-learn \
    sentencepiece \
    safetensors



     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.8/43.8 kB 1.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.1/9.1 MB 54.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 251.6/251.6 kB 16.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 302.6/302.6 kB 18.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 24.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.6/3.6 MB 89.3 MB/s eta 0:00:00


In [ ]:
# ─── 2. SETUP & IMPORTS ───────────────────────────────────────────────────────
import os, time, functools, json
import torch
import numpy as np
import pandas as pd
import torch.nn as nn
from PIL import Image
from datasets import load_dataset, Image as HFImage
from sklearn.metrics import accuracy_score, precision_recall_fscore_support, roc_auc_score
from transformers import (CLIPProcessor, CLIPModel, TrainingArguments,
                          Trainer, EarlyStoppingCallback)
from transformers.modeling_outputs import SequenceClassifierOutput
from transformers.trainer_utils import get_last_checkpoint
from google.colab import drive

# Tối ưu bộ nhớ GPU
os.environ["PYTORCH_ALLOC_CONF"] = "expandable_segments:True"
device = "cuda" if torch.cuda.is_available() else "cpu"
print("Using device:", device)

# Kết nối Drive
drive.mount("/content/drive")
SAVE_DIR = "/content/drive/MyDrive/CLIP_PromptTuning_MirageNews_Final"
os.makedirs(SAVE_DIR, exist_ok=True)
print("Save to:", SAVE_DIR)

# ─── 3. DATASET & PREPROCESS ──────────────────────────────────────────────────
print("⏳ Loading dataset MirageNews...")
dataset = load_dataset("anson-huang/mirage-news")
dataset = dataset.cast_column("image", HFImage(decode=True))
dataset = dataset.rename_column("label", "labels")

model_name = "openai/clip-vit-base-patch32"
processor = CLIPProcessor.from_pretrained(model_name)
PROMPT_LENGTH = 5 # Độ dài Soft Prompt

def preprocess(examples):
    # Đảm bảo ảnh ở định dạng RGB
    images = [img.convert("RGB") if img.mode != "RGB" else img for img in examples["image"]]

    inputs = processor(
        text=examples["text"],
        images=images,
        padding="max_length",
        truncation=True,
        max_length=77 - PROMPT_LENGTH
    )
    inputs["labels"] = examples["labels"]
    return inputs

print("⏳ Preprocessing dataset (IND & OOD splits)...")
encoded_dataset = dataset.map(
    preprocess,
    remove_columns=dataset["train"].column_names,
    batched=True,
    desc="Preprocessing"
)

encoded_dataset.set_format(
    type="torch",
    columns=["input_ids", "attention_mask", "pixel_values", "labels"]
)
print("✅ Preprocessing hoàn tất.")

# ─── 4. MODEL ARCHITECTURE (PROMPT TUNING + GATED FUSION + ALIGNMENT) ────────
class CLIPPromptTuningForMFND(nn.Module):
    def __init__(self, model_name="openai/clip-vit-base-patch32", num_labels=2,
                 prompt_length=5, lambda_weight=0.1, delta=0.5):
        super().__init__()

        self.clip = CLIPModel.from_pretrained(model_name)
        self.prompt_length = prompt_length

        # 1. Đóng băng Base Model
        for param in self.clip.parameters():
            param.requires_grad = False

        self.hidden_dim = self.clip.config.text_config.hidden_size
        embed_dim = self.clip.config.projection_dim # 512

        # 2. Khởi tạo Soft Prompt (Học được)
        self.soft_prompt = nn.Parameter(torch.randn(prompt_length, self.hidden_dim))

        # 3. Gated Fusion & Alignment Head
        self.cross_attention = nn.MultiheadAttention(embed_dim=embed_dim, num_heads=8, batch_first=True)
        self.W_g = nn.Linear(embed_dim * 2, embed_dim)
        self.classifier = nn.Linear(embed_dim, num_labels)

        # Mở khóa Gradient cho các thành phần mới
        for param in self.cross_attention.parameters(): param.requires_grad = True
        for param in self.W_g.parameters(): param.requires_grad = True
        for param in self.classifier.parameters(): param.requires_grad = True

        self.lambda_weight = lambda_weight
        self.delta = delta
        self.align_loss_fn = nn.CosineEmbeddingLoss(margin=delta)

    def forward(self, input_ids, attention_mask, pixel_values, labels=None):
        batch_size = input_ids.size(0)
        device = input_ids.device

        # --- TEXT PATHWAY (Injecting Soft Prompt) ---
        token_embeddings = self.clip.text_model.embeddings.token_embedding(input_ids)
        soft_prompt = self.soft_prompt.unsqueeze(0).expand(batch_size, -1, -1)
        hidden_states = torch.cat([soft_prompt, token_embeddings], dim=1)

        prompt_mask = torch.ones(batch_size, self.prompt_length, device=device)
        attention_mask_text = torch.cat([prompt_mask, attention_mask], dim=1)

        seq_length = hidden_states.size(1)
        position_ids = torch.arange(seq_length, device=device).unsqueeze(0)
        position_embeddings = self.clip.text_model.embeddings.position_embedding(position_ids)
        hidden_states = hidden_states + position_embeddings

        causal_attention_mask = torch.full((seq_length, seq_length), float("-inf"), device=device)
        causal_attention_mask = torch.triu(causal_attention_mask, diagonal=1)
        causal_attention_mask = causal_attention_mask.unsqueeze(0).unsqueeze(0).expand(batch_size, 1, seq_length, seq_length)

        if attention_mask_text is not None:
            attention_mask_text = attention_mask_text[:, None, None, :].expand(batch_size, 1, seq_length, seq_length)

        encoder_outputs = self.clip.text_model.encoder(
            inputs_embeds=hidden_states,
            attention_mask=attention_mask_text,
            causal_attention_mask=causal_attention_mask,
        )

        hidden_states = encoder_outputs[0]
        hidden_states = self.clip.text_model.final_layer_norm(hidden_states)

        # Lấy EOS token embedding (đã bị dịch chuyển bởi prompt)
        eos_token_id = self.clip.config.text_config.eos_token_id
        eos_mask = (input_ids == eos_token_id)
        eos_positions = eos_mask.float().argmax(dim=1) + self.prompt_length

        pooled_output = hidden_states[torch.arange(batch_size, device=device), eos_positions]
        text_features = self.clip.text_projection(pooled_output)
        h_T = text_features / text_features.norm(dim=-1, keepdim=True)

        # --- IMAGE PATHWAY ---
        image_outputs = self.clip.vision_model(pixel_values=pixel_values)
        image_features = self.clip.visual_projection(image_outputs.pooler_output)
        h_I = image_features / image_features.norm(dim=-1, keepdim=True)

        # --- GATED MODALITY FUSION ---
        attn_out, _ = self.cross_attention(h_T.unsqueeze(1), h_I.unsqueeze(1), h_I.unsqueeze(1))
        z = attn_out.squeeze(1)

        gate = torch.sigmoid(self.W_g(torch.cat([h_T, h_I], dim=1)))
        h_f = gate * z

        # --- CLASSIFICATION & ALIGNMENT LOSS ---
        logits = self.classifier(h_f)
        loss = None
        if labels is not None:
            loss_cls = nn.CrossEntropyLoss()(logits, labels)
            # Alignment Logic: 1 - 2*label (Khớp MirageNews & IFND)
            target = 1 - (labels.float() * 2)
            loss_align = self.align_loss_fn(h_T, h_I, target)
            loss = loss_cls + (self.lambda_weight * loss_align)

        return SequenceClassifierOutput(loss=loss, logits=logits)

import gc; gc.collect(); torch.cuda.empty_cache()
model = CLIPPromptTuningForMFND(prompt_length=PROMPT_LENGTH).to(device)

total_params     = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"\n📊 Total params    : {total_params:,}")
print(f"📊 Trainable params: {trainable_params:,} ({trainable_params/total_params*100:.4f}%)")

# ─── 5. METRICS & TRAINING ────────────────────────────────────────────────────
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    probs  = torch.softmax(torch.tensor(logits), dim=1).numpy()
    preds  = np.argmax(probs, axis=1)
    p, r, f1, _ = precision_recall_fscore_support(labels, preds, average="binary")
    return {
        "accuracy":  accuracy_score(labels, preds),
        "precision": p,
        "recall":    r,
        "f1":        f1,
        "auc":       roc_auc_score(labels, probs[:, 1])
    }

torch.load = functools.partial(torch.load, weights_only=False)

training_args = TrainingArguments(
    output_dir=SAVE_DIR,
    overwrite_output_dir=True,
    num_train_epochs=5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=32,
    gradient_accumulation_steps=1,
    learning_rate=1e-3, # Prompt Tuning thường dùng LR cao (1e-3)
    weight_decay=0.01,
    warmup_ratio=0.1,
    evaluation_strategy="epoch",
    save_strategy="epoch",
    save_total_limit=1,
    load_best_model_at_end=True,
    metric_for_best_model="f1",
    fp16=torch.cuda.is_available(),
    dataloader_num_workers=0,
    report_to="none"
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=encoded_dataset["train"],
    eval_dataset=encoded_dataset["validation"],
    compute_metrics=compute_metrics,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=2)]
)

print("\n🚀 Bắt đầu huấn luyện Prompt Tuning...")
if torch.cuda.is_available(): torch.cuda.reset_peak_memory_stats()
start_train = time.time()
trainer.train()

trainer.save_model(SAVE_DIR)
processor.save_pretrained(SAVE_DIR)
train_time_min = (time.time() - start_train) / 60
print(f"✅ Training hoàn tất: {train_time_min:.2f} phút")

# ─── 6. INFERENCE MEASUREMENT (BATCH SIZE = 1, P50 MEDIAN) ────────────────────
model.eval()
dummy_image = Image.new("RGB", (224, 224), (128, 128, 128))
dummy_text  = "Sample news for latency measurement."
inputs = processor(text=dummy_text, images=dummy_image, truncation=True, padding="max_length", max_length=77-PROMPT_LENGTH, return_tensors="pt")
input_ids = inputs["input_ids"].to(device)
attention_mask = inputs["attention_mask"].to(device)
pixel_values = inputs["pixel_values"].to(device)

# Warmup
with torch.no_grad():
    for _ in range(50): model(input_ids=input_ids, attention_mask=attention_mask, pixel_values=pixel_values)

latencies = []
with torch.no_grad():
    for _ in range(200):
        if device == "cuda": torch.cuda.synchronize()
        t0 = time.perf_counter()
        model(input_ids=input_ids, attention_mask=attention_mask, pixel_values=pixel_values)
        if device == "cuda": torch.cuda.synchronize()
        latencies.append((time.perf_counter() - t0) * 1000)

latencies = np.array(latencies)
vram = torch.cuda.max_memory_allocated() / 1024**3 if device == "cuda" else 0

# ─── 7. FINAL EVALUATION & SAVE RESULTS ───────────────────────────────────────
print("\n📊 Đang đánh giá trên toàn bộ các tập test của MirageNews...")
test_splits = ["test1_nyt_mj", "test2_bbc_dalle", "test3_cnn_dalle", "test4_bbc_sdxl", "test5_cnn_sdxl"]

final_results = {
    "Model": "CLIP ViT-B/32",
    "Method": "Prompt Tuning (Length=5)",
    "Dataset": "MirageNews",
    "Hardware_Stats": {
        "Trainable_Params": int(trainable_params),
        "Training_Time_Min": round(train_time_min, 2),
        "Peak_VRAM_GB": round(vram, 2),
        "Latency_P50_ms": round(np.median(latencies), 2),
        "Latency_P90_ms": round(np.percentile(latencies, 90), 2)
    },
    "Test_Splits": {}
}

ood_accumulator = {"Accuracy": [], "Precision": [], "Recall": [], "F1": [], "AUC": []}

for split in test_splits:
    out = trainer.predict(encoded_dataset[split])
    m = out.metrics

    metrics = {
        "Accuracy": round(m["test_accuracy"] * 100, 2),
        "Precision": round(m["test_precision"] * 100, 2),
        "Recall": round(m["test_recall"] * 100, 2),
        "F1": round(m["test_f1"] * 100, 2),
        "AUC": round(m["test_auc"], 4)
    }
    final_results["Test_Splits"][split] = metrics

    if split != "test1_nyt_mj": # Tính trung bình OOD cho Test 2-5
        for k in ood_accumulator.keys(): ood_accumulator[k].append(metrics[k])

final_results["OOD_Average"] = {k: round(np.mean(v), 2) for k, v in ood_accumulator.items()}

# Lưu file JSON
json_path = os.path.join(SAVE_DIR, "results_PromptTuning.json")
with open(json_path, "w") as f:
    json.dump(final_results, f, indent=4)

print(f"\n✅ Đã lưu kết quả tại: {json_path}")
print(f"OOD Average F1: {final_results['OOD_Average']['F1']}%")
print(f"Latency P50: {final_results['Hardware_Stats']['Latency_P50_ms']} ms")
print(f"{'='*50}")

# ─── 8. AUTO-DISCONNECT GPU ───────────────────────────────────────────────────
print("⏳ Đang ngắt kết nối GPU...")
from google.colab import runtime
time.sleep(10) # Đợi đồng bộ Drive
runtime.unassign()

Using device: cuda
Mounted at /content/drive
Save to: /content/drive/MyDrive/CLIP_PromptTuning_MirageNews_Final
⏳ Loading dataset MirageNews...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


README.md: 0.00B [00:00, ?B/s]

data/train-00000-of-00001.parquet:   0%|          | 0.00/655M [00:00<?, ?B/s]

data/validation-00000-of-00001.parquet:   0%|          | 0.00/143M [00:00<?, ?B/s]

...


📊 Total params    : 152,856,323
📊 Trainable params: 1,579,010 (1.0330%)

🚀 Bắt đầu huấn luyện Prompt Tuning...


/usr/local/lib/python3.12/dist-packages/transformers/training_args.py:1474: FutureWarning: `evaluation_strategy` is deprecated and will be removed in version 4.46 of 🤗 Transformers. Use `eval_strategy` instead
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/accelerate/accelerator.py:479: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  self.scaler = torch.cuda.amp.GradScaler(**kwargs)


Epoch,Training Loss,Validation Loss,Accuracy,Precision,Recall,F1,Auc
1,0.309300,0.150153,0.954400,0.952229,0.956800,0.954509,0.992292
2,0.151000,0.139019,0.962000,0.972200,0.951200,0.961585,0.993904
3,0.137400,0.124272,0.966000,0.962669,0.969600,0.966122,0.995140
4,0.106500,0.130385,0.964400,0.956009,0.973600,0.964725,0.995049
5,0.095100,0.125613,0.967200,0.967200,0.967200,0.967200,0.995451


✅ Training hoàn tất: 20.38 phút

📊 Đang đánh giá trên toàn bộ các tập test của MirageNews...



✅ Đã lưu kết quả tại: /content/drive/MyDrive/CLIP_PromptTuning_MirageNews_Final/results_PromptTuning.json
OOD Average F1: 67.57%
Latency P50: 26.75 ms
⏳ Đang ngắt kết nối GPU...


In [ ]:
import json
from google.colab import drive

# 1. Kết nối Google Drive (Bỏ qua nếu bạn đã mount rồi)
drive.mount('/content/drive')

# 2. Đường dẫn chính xác đến file JSON của bạn
file_path = '/content/drive/MyDrive/CLIP_PromptTuning_MirageNews_Final/results_PromptTuning.json'

# 3. Đọc và phân tích file
try:
    with open(file_path, 'r', encoding='utf-8') as f:
        results_data = json.load(f)

    # In toàn bộ dữ liệu dưới dạng JSON format (thụt lề 4 ô cho dễ đọc)
    print("📊 TOÀN BỘ NỘI DUNG FILE KẾT QUẢ:\n")
    print(json.dumps(results_data, indent=4, ensure_ascii=False))

    # --- Tùy chọn: Trích xuất nhanh các thông số quan trọng để làm báo cáo ---
    print("\n" + "="*50)
    print("🎯 TÓM TẮT NHANH CHỈ SỐ QUAN TRỌNG:")

    # Trích xuất phần cứng
    hw = results_data.get("Hardware_Stats", {})
    print(f"⚡ VRAM Peak       : {hw.get('Peak_VRAM_GB', 'N/A')} GB")
    print(f"⚡ Latency P50     : {hw.get('Latency_P50_ms', 'N/A')} ms")
    print(f"⚡ Trainable Params: {hw.get('Trainable_Params', 'N/A'):,}")

    # Trích xuất hiệu suất OOD
    ood = results_data.get("OOD_Average", {})
    if ood:
        print(f"\n📈 OOD F1 Average  : {ood.get('F1', 'N/A')}%")
        print(f"📈 OOD Accuracy Avg: {ood.get('Accuracy', 'N/A')}%")

    print("="*50)

except FileNotFoundError:
    print(f"❌ Không tìm thấy file tại đường dẫn: {file_path}")
    print("Vui lòng kiểm tra lại tên thư mục hoặc đảm bảo file đã được lưu thành công ở bước trước.")
except json.JSONDecodeError:
    print("❌ File được tìm thấy nhưng bị lỗi định dạng JSON. Vui lòng kiểm tra lại nội dung file.")

Mounted at /content/drive
📊 TOÀN BỘ NỘI DUNG FILE KẾT QUẢ:

{
    "Model": "CLIP ViT-B/32",
    "Method": "Prompt Tuning (Length=5)",
    "Dataset": "MirageNews",
    "Hardware_Stats": {
        "Trainable_Params": 1579010,
        "Training_Time_Min": 20.38,
        "Peak_VRAM_GB": 0.97,
        "Latency_P50_ms": 26.75,
        "Latency_P90_ms": 48.31
    },
    "Test_Splits": {
        "test1_nyt_mj": {
            "Accuracy": 97.8,
            "Precision": 98.78,
            "Recall": 96.8,
            "F1": 97.78,
            "AUC": 0.998
        },
        "test2_bbc_dalle": {
            "Accuracy": 61.2,
            "Precision": 64.29,
            "Recall": 50.4,
            "F1": 56.5,
            "AUC": 0.6903
        },
        "test3_cnn_dalle": {
            "Accuracy": 65.6,
            "Precision": 70.53,
            "Recall": 53.6,
            "F1": 60.91,
            "AUC": 0.7062
        },
        "test4_bbc_sdxl": {
            "Accuracy": 73.6,
            "Precisio